# Task 1: Vectorized Scaled Dot-Product Attention from Scratch

**Objective:** Derive and implement the core self-attention mechanism used in Transformer
models, using only raw NumPy linear algebra — no deep learning frameworks.

**Covers:**
- Batch-wise processing of multi-head inputs (4D tensors: `[batch, heads, seq_len, d_k]`)
- Causal (look-ahead) masking for decoder-style attention
- Correct scaling by $\sqrt{d_k}$
- A numerically stable softmax implemented from scratch

**Tech stack:** Pure NumPy, Python 3.10+, Jupyter Notebook.


## 1. Imports

In [1]:
import numpy as np

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## 2. The Math

For a single head, given queries $Q$, keys $K$, and values $V$:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V
$$

- $Q \in \mathbb{R}^{n \times d_k}$, $K \in \mathbb{R}^{n \times d_k}$, $V \in \mathbb{R}^{n \times d_v}$
- $d_k$ is the key/query dimension. Scaling by $\sqrt{d_k}$ keeps the dot products from
  growing too large in magnitude as $d_k$ increases, which would otherwise push the
  softmax into regions with vanishing gradients.
- $M$ is an additive mask. For causal (decoder) attention, $M_{ij} = -\infty$ whenever
  $j > i$, so position $i$ can never attend to a future position $j$.

We implement this so it works on a **4D batch of multi-head tensors**
`[batch_size, num_heads, seq_len, d_k]` all at once, using NumPy's batched matrix
multiplication (`@` broadcasts over the leading dimensions).


## 3. Numerically Stable Softmax

We subtract the row-wise max before exponentiating to avoid `inf`/`nan` from large logits.

In [2]:
def softmax(x, axis=-1):
    """Numerically stable softmax along the given axis.

    x: np.ndarray of arbitrary shape.
    Returns an array of the same shape, summing to 1 along `axis`.
    """
    x_max = np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x - x_max)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)


# Quick sanity check
test = np.array([[1.0, 2.0, 3.0], [1000.0, 1000.0, 1001.0]])
probs = softmax(test)
print(probs)
print("Row sums:", probs.sum(axis=-1))  # should be [1., 1.]

[[0.09   0.2447 0.6652]
 [0.2119 0.2119 0.5761]]
Row sums: [1. 1.]


## 4. Causal Mask Construction

Builds an additive mask of shape `[seq_len, seq_len]` where entry `(i, j)` is `0` if
`j <= i` (allowed) and `-inf` if `j > i` (future position, forbidden).

In [3]:
def build_causal_mask(seq_len: int) -> np.ndarray:
    """Additive causal mask of shape [seq_len, seq_len].

    Upper triangle (excluding diagonal) is -inf so softmax zeroes those
    positions out; lower triangle (including diagonal) is 0.
    """
    mask = np.triu(np.ones((seq_len, seq_len)), k=1) * -np.inf
    # np.triu with -inf * 0 on the diagonal/lower part produces nan, so rebuild cleanly:
    mask = np.where(np.triu(np.ones((seq_len, seq_len)), k=1) == 1, -np.inf, 0.0)
    return mask


print(build_causal_mask(5))

[[  0. -inf -inf -inf -inf]
 [  0.   0. -inf -inf -inf]
 [  0.   0.   0. -inf -inf]
 [  0.   0.   0.   0. -inf]
 [  0.   0.   0.   0.   0.]]


/tmp/ipykernel_838/51043022.py:7: RuntimeWarning: invalid value encountered in multiply
  mask = np.triu(np.ones((seq_len, seq_len)), k=1) * -np.inf


## 5. Vectorized Scaled Dot-Product Attention

Operates on 4D tensors `[batch, heads, seq_len, d_k]`. This is exactly the shape used
inside a real multi-head attention layer after splitting the embedding dimension
across heads.

In [4]:
def scaled_dot_product_attention(Q, K, V, causal=False, mask=None):
    """Vectorized scaled dot-product attention over batched, multi-head inputs.

    Parameters
    ----------
    Q, K, V : np.ndarray
        Shape [batch, heads, seq_len, d_k] (Q and K must share d_k; V's last dim
        can differ, conventionally called d_v).
    causal : bool
        If True, applies a causal (look-ahead) mask so position i cannot
        attend to position j > i. Used to simulate decoder self-attention.
    mask : np.ndarray or None
        Optional additive mask broadcastable to [batch, heads, seq_len, seq_len]
        (e.g. a padding mask). Combined with the causal mask if both given.

    Returns
    -------
    output : np.ndarray, shape [batch, heads, seq_len, d_v]
    attn_weights : np.ndarray, shape [batch, heads, seq_len, seq_len]
    """
    d_k = Q.shape[-1]

    # QK^T over the last two dims, batched over [batch, heads]
    # Q: [B, H, N, d_k], K: [B, H, N, d_k] -> scores: [B, H, N, N]
    scores = Q @ np.swapaxes(K, -1, -2)
    scores = scores / np.sqrt(d_k)

    if causal:
        seq_len = Q.shape[-2]
        causal_mask = build_causal_mask(seq_len)  # [N, N], broadcasts over [B, H, N, N]
        scores = scores + causal_mask

    if mask is not None:
        scores = scores + mask

    attn_weights = softmax(scores, axis=-1)  # [B, H, N, N]
    output = attn_weights @ V  # [B, H, N, N] @ [B, H, N, d_v] -> [B, H, N, d_v]

    return output, attn_weights

## 6. Multi-Head Wrapper

Splits an embedding of size `d_model` into `num_heads` heads of size
`d_k = d_model / num_heads`, runs attention per head, then concatenates and
projects back — the standard multi-head attention block.

In [5]:
class MultiHeadAttention:
    """Multi-head scaled dot-product attention, implemented with raw NumPy.

    Learnable projection matrices are randomly initialized (this is a from-scratch
    forward-pass implementation, not a trained module).
    """

    def __init__(self, d_model: int, num_heads: int, seed: int = 0):
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        rng = np.random.default_rng(seed)
        scale = 1.0 / np.sqrt(d_model)
        self.W_q = rng.normal(0, scale, size=(d_model, d_model))
        self.W_k = rng.normal(0, scale, size=(d_model, d_model))
        self.W_v = rng.normal(0, scale, size=(d_model, d_model))
        self.W_o = rng.normal(0, scale, size=(d_model, d_model))

    def _split_heads(self, x):
        # x: [batch, seq_len, d_model] -> [batch, heads, seq_len, d_k]
        batch, seq_len, _ = x.shape
        x = x.reshape(batch, seq_len, self.num_heads, self.d_k)
        return x.transpose(0, 2, 1, 3)

    def _combine_heads(self, x):
        # x: [batch, heads, seq_len, d_k] -> [batch, seq_len, d_model]
        batch, heads, seq_len, d_k = x.shape
        x = x.transpose(0, 2, 1, 3)
        return x.reshape(batch, seq_len, heads * d_k)

    def forward(self, x, causal=False):
        """x: [batch, seq_len, d_model]"""
        Q = x @ self.W_q
        K = x @ self.W_k
        V = x @ self.W_v

        Q, K, V = self._split_heads(Q), self._split_heads(K), self._split_heads(V)

        out, attn_weights = scaled_dot_product_attention(Q, K, V, causal=causal)

        out = self._combine_heads(out)
        out = out @ self.W_o
        return out, attn_weights

## 7. Demo: Batched Multi-Head Self-Attention with Causal Masking

In [6]:
batch_size, seq_len, d_model, num_heads = 2, 6, 16, 4

x = np.random.randn(batch_size, seq_len, d_model)

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads, seed=42)
output, attn_weights = mha.forward(x, causal=True)

print("Input shape: ", x.shape)
print("Output shape:", output.shape)
print("Attention weights shape:", attn_weights.shape)  # [batch, heads, seq_len, seq_len]

# Confirm causal masking worked: for head 0, batch 0, the weights above the
# diagonal should all be ~0
print("\nAttention map for batch 0, head 0 (rows=query pos, cols=key pos):")
print(np.round(attn_weights[0, 0], 3))

Input shape:  (2, 6, 16)
Output shape: (2, 6, 16)
Attention weights shape: (2, 4, 6, 6)

Attention map for batch 0, head 0 (rows=query pos, cols=key pos):
[[1.    0.    0.    0.    0.    0.   ]
 [0.347 0.653 0.    0.    0.    0.   ]
 [0.201 0.287 0.512 0.    0.    0.   ]
 [0.234 0.219 0.252 0.295 0.    0.   ]
 [0.221 0.042 0.083 0.373 0.281 0.   ]
 [0.126 0.062 0.042 0.205 0.385 0.18 ]]


/tmp/ipykernel_838/51043022.py:7: RuntimeWarning: invalid value encountered in multiply
  mask = np.triu(np.ones((seq_len, seq_len)), k=1) * -np.inf


## 8. Correctness Checks

In [7]:
# 1) Each row of the attention map must sum to 1 (softmax property)
row_sums = attn_weights.sum(axis=-1)
assert np.allclose(row_sums, 1.0), "Attention rows must sum to 1"
print("Row sums all ~1.0: OK")

# 2) Causal property: weight assigned to any future key position must be ~0
future_mask = np.triu(np.ones((seq_len, seq_len)), k=1).astype(bool)
future_weights = attn_weights[:, :, future_mask]
assert np.allclose(future_weights, 0.0, atol=1e-6), "Causal mask leaked future info"
print("No attention leaked to future positions: OK")

# 3) Shape sanity
assert output.shape == (batch_size, seq_len, d_model)
print("Output shape matches [batch, seq_len, d_model]: OK")

# 4) Compare against a naive, non-vectorized loop implementation (single head, single batch)
def naive_attention_single_head(Q, K, V, causal=True):
    n, d_k = Q.shape
    out = np.zeros_like(V)
    weights = np.zeros((n, n))
    for i in range(n):
        scores = np.zeros(n)
        limit = i + 1 if causal else n
        for j in range(limit):
            scores[j] = np.dot(Q[i], K[j]) / np.sqrt(d_k)
        w = softmax(scores[:limit])
        weights[i, :limit] = w
        out[i] = w @ V[:limit]
    return out, weights

Qh = x[0] @ mha.W_q
Kh = x[0] @ mha.W_k
Vh = x[0] @ mha.W_v
Qh_split = mha._split_heads(Qh[None])[0, 0]  # head 0
Kh_split = mha._split_heads(Kh[None])[0, 0]
Vh_split = mha._split_heads(Vh[None])[0, 0]

naive_out, naive_w = naive_attention_single_head(Qh_split, Kh_split, Vh_split, causal=True)
assert np.allclose(naive_out, attn_weights[0, 0] @ Vh_split, atol=1e-6)
assert np.allclose(naive_w, attn_weights[0, 0], atol=1e-6)
print("Vectorized implementation matches naive loop implementation: OK")

Row sums all ~1.0: OK
No attention leaked to future positions: OK
Output shape matches [batch, seq_len, d_model]: OK
Vectorized implementation matches naive loop implementation: OK


## 9. Summary

- `softmax` — numerically stable softmax used throughout.
- `build_causal_mask` — additive `[-inf, 0]` mask enforcing autoregressive decoding.
- `scaled_dot_product_attention` — fully vectorized over `[batch, heads, seq_len, d_k]`,
  matching the formula $\text{softmax}(QK^\top/\sqrt{d_k} + M)V$.
- `MultiHeadAttention` — splits/combines heads and wraps the core attention function
  with learnable (randomly initialized) linear projections.
- Verified correctness against a naive nested-loop reference implementation and confirmed
  the causal mask prevents any leakage of future-position information.
